In [1]:
import os
import requests
import sys
import time

from SPARQLWrapper import SPARQLWrapper, JSON, POST, DIGEST, POSTDIRECTLY
import base64

from requests.auth import HTTPDigestAuth, HTTPBasicAuth


#from importlib import reload

## https://anaconda.org/channels/conda-forge/packages/python-dotenv/overview
from dotenv import load_dotenv

In [2]:
from rdflib import Graph, Namespace, URIRef
from rdflib.namespace import RDF, RDFS, XSD

In [3]:

# Add parent directory to the path
sys.path.insert(0, '..')

### If you want to add the parent-parent directory,
sys.path.insert(0, '../..')

## Documentation

* [Documentation du processus](https://euria.infomaniak.com/shared/019ea929-55bd-73d0-96c7-021c37d32feb)
* [SPARQLwrapper documentation](https://sparqlwrapper.readthedocs.io/en/latest/main.html)

In [4]:
# Load variables from .env file into environment
load_dotenv(override=True)

# Retrieve variables (returns None if not found)
SOURCE_URL = os.getenv("SOURCE_URL")
SOURCE_GRAPH = os.getenv("SOURCE_GRAPH")
TARGET_URL = os.getenv("TARGET_URL")
TARGET_URL_SELECT = os.getenv("TARGET_URL_SELECT")
TARGET_GRAPH = os.getenv("TARGET_GRAPH")

TARGET_USER = os.getenv("TARGET_USER")
TARGET_PASS = os.getenv("TARGET_PASS")


In [5]:
print(TARGET_URL,TARGET_GRAPH)

https://swiss-elites.lod4hss.cloud/wisski/endpoint/default/statements https://swiss-elites.lod4hss.cloud/resource/


In [6]:
def count_quads(source_url, graph_uri=None):
    """Count quads in the source graph using a SELECT query."""
    if graph_uri:
        where = f"GRAPH <{graph_uri}> {{ ?s ?p ?o ?g }}"
        # Note: Counting quads specifically requires ?g in some stores, or just count triples in that graph
        # Standard way to count triples in a named graph:
        where = f"GRAPH <{graph_uri}> {{ ?s ?p ?o }}"
    else:
        where = "{ ?s ?p ?o }"
        
    query = f"SELECT (COUNT(*) as ?c) WHERE {{ {where} }}"
    # print(query)
    
    try:
        r = requests.get(source_url, params={"query": query}, headers={"Accept": "application/sparql-results+json"}, auth=AUTH, timeout=TIMEOUT)
        r.raise_for_status()
        return int(r.json()['results']['bindings'][0]['c']['value'])
    except Exception as e:
        print(f"Warning: Could not get count. {e}")
        return None

In [8]:
TIMEOUT = 60

In [9]:
AUTH=None
print(count_quads(SOURCE_URL))

315078


## Test if the INSERT works

In [33]:

# Setup the endpoint
sparql_insert = SPARQLWrapper(TARGET_URL)

# not working with RDF4J
# sparql.setHTTPAuth(DIGEST)

sparql_insert.setCredentials(TARGET_USER, TARGET_PASS)

# Configure for UPDATE (INSERT/DELETE)
sparql_insert.setMethod(POST)

# not needed
#sparql.setRequestMethod(POSTDIRECTLY)  # required for updates


In [32]:

# 4. Define the SPARQL INSERT query
# Note: Always use full URIs in < > or prefixed names if prefixes are defined
query = """
INSERT DATA {
 GRAPH <http://elites-suisses.org/test> {
  <http://example.org/person/luke> <http://xmlns.com/foaf/0.1/name> "Luke Luky" .
  <http://example.org/person/luke> <http://xmlns.com/foaf/0.1/knows> <http://example.org/person/jane> .
}
}
"""


In [ ]:
query="""
INSERT DATA {
        GRAPH <http://elites-suisses.org/test> {
        <https://swiss-elites.lod4hss.cloud/resource/bir_100047> <https://sdhss.org/ontology/shortcuts/P1> "1789"^^<http://www.w3.org/2001/XMLSchema#integer> .
<https://swiss-elites.lod4hss.cloud/resource/bir_100047> <http://www.cidoc-crm.org/cidoc-crm/P97> <https://swiss-elites.lod4hss.cloud/resource/p100946> .
<https://swiss-elites.lod4hss.cloud/resource/bir_100047> <http://www.cidoc-crm.org/cidoc-crm/P98> <https://swiss-elites.lod4hss.cloud/resource/p100047> .
<https://swiss-elites.lod4hss.cloud/resource/bir_71787> <http://www.cidoc-crm.org/cidoc-crm/P98> <https://swiss-elites.lod4hss.cloud/resource/p71787> .
<https://swiss-elites.lod4hss.cloud/resource/bir_71787> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.cidoc-crm.org/cidoc-crm/E67> .
<https://swiss-elites.lod4hss.cloud/resource/bir_71794> <https://sdhss.org/ontology/shortcuts/P1> "1954"^^<http://www.w3.org/2001/XMLSchema#integer> .
<https://swiss-elites.lod4hss.cloud/resource/bir_71794> <https://sdhss.org/ontology/social-life-core/P20> <https://swiss-elites.lod4hss.cloud/resource/p66705> .
<https://swiss-elites.lod4hss.cloud/resource/mar6670566706> <https://sdhss.org/ontology/social-life-core/P20> <https://swiss-elites.lod4hss.cloud/resource/p66706> .
 <https://swiss-elites.lod4hss.cloud/resource/bir_71787> <http://www.cidoc-crm.org/cidoc-crm/P98> <https://swiss-elites.lod4hss.cloud/resource/p71787> .
<https://swiss-elites.lod4hss.cloud/resource/bir_71787> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.cidoc-crm.org/cidoc-crm/E67> .
<https://swiss-elites.lod4hss.cloud/resource/bir_71794> <https://sdhss.org/ontology/shortcuts/P1> "1954"^^<http://www.w3.org/2001/XMLSchema#integer> .
<https://swiss-elites.lod4hss.cloud/resource/mar6670566706> <https://sdhss.org/ontology/social-life-core/P20> <https://swiss-elites.lod4hss.cloud/resource/p66706> .
<https://swiss-elites.lod4hss.cloud/resource/bir_71768> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.cidoc-crm.org/cidoc-crm/E67> .
<https://swiss-elites.lod4hss.cloud/resource/bir_71787> <https://sdhss.org/ontology/shortcuts/P1> "1949"^^<http://www.w3.org/2001/XMLSchema#integer> .
<https://swiss-elites.lod4hss.cloud/resource/bir_71787> <http://www.cidoc-crm.org/cidoc-crm/P97> <https://swiss-elites.lod4hss.cloud/resource/p73653> .   
}
}
"""

In [34]:

sparql_insert.setQuery(query)

# 5. Execute
try:
    out=sparql_insert.query()
    print(out.response.status)
    print("✓ Success: Triple(s) inserted.")
except Exception as e:
    print(f"✗ Error: {e}")

✗ Error: HTTP Error 403: Forbidden


In [98]:
print(dir(sparql_insert.query().response))

['__abstractmethods__', '__class__', '__del__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__next__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '_abc_impl', '_checkClosed', '_checkReadable', '_checkSeekable', '_checkWritable', '_check_close', '_close_conn', '_get_chunk_left', '_method', '_peek_chunked', '_read1_chunked', '_read_and_discard_trailer', '_read_chunked', '_read_next_chunk_size', '_read_status', '_readinto_chunked', '_safe_read', '_safe_readinto', 'begin', 'chunk_left', 'chunked', 'close', 'closed', 'code', 'debuglevel', 'detach', 'fileno', 'flush', 'fp', 'getcode', 'getheader', 'getheaders', 'geturl', 'headers', 'info', 'isatty', 'isclosed', 'length', 'msg', 'peek', 'read', 'read1', 'readable',

In [13]:
print(TARGET_URL)

https://open-gdb.lod4hss.org/repositories/eos/statements


In [55]:
query="""
SELECT * 
WHERE { ?s <http://xmlns.com/foaf/0.1/knows> <http://example.org/person/jane> }
"""

In [56]:
target_select = SPARQLWrapper(TARGET_URL_SELECT)
target_select.setMethod("GET") # Reset to GET for reading
target_select.setQuery(query)
target_select.setReturnFormat(JSON)
print(target_select.query().convert())

{'head': {'vars': ['s']}, 'results': {'bindings': [{'s': {'type': 'uri', 'value': 'http://example.org/person/john'}}, {'s': {'type': 'uri', 'value': 'http://example.org/person/luke'}}]}}


## Batch insert

In [18]:

def fetch_batch(source_endpoint, offset, limit, auth_header=None):
    """Fetches a batch of triples from the source."""
    sparql = SPARQLWrapper(source_endpoint)
    
    # Query to get raw triples
    query = f"""
    SELECT ?s ?p ?o
    WHERE {{
      ?s ?p ?o .
    }}
    ORDER BY ?s ?p ?o
    LIMIT {limit}
    OFFSET {offset}
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    if auth_header:
        sparql.addRequestHeader("Authorization", auth_header)
    
    try:
        results = sparql.query().convert()
        return results["results"]["bindings"]
    except Exception as e:
        print(f"Error fetching batch at offset {offset}: {e}")
        return []


In [19]:

def format_triple(binding):
    """Converts a JSON binding result into a SPARQL triple string."""
    s_val = binding['s']['value'].strip()
    p_val = binding['p']['value'].strip()
    o_val = binding['o']['value'].strip()
    o_type = binding['o']['type']


    # Encode problematic characters in URIs
    def safe_uri(uri):
        # Strip whitespace, encode spaces if any
        uri = uri.strip().replace(" ", "%20").replace("\n", "").replace("\r", "")
        return f"<{uri}>"



    
    # Format Subject
    s = safe_uri(s_val)
    
    # Format Predicate
    p = safe_uri(p_val)
    
    # Format Object (handle URI vs Literal)
    if o_type == 'uri':
        o = safe_uri(o_val)
    else:
        # Handle Literals
        datatype = binding['o'].get('datatype')
        lang = binding['o'].get('xml:lang')
        
        # Escape quotes in literal values just in case
        safe_val = o_val.replace('\\', '\\\\').replace('"', '\\"')
        
        if datatype:
            o = f'"{safe_val}"^^<{datatype}>'
        elif lang:
            o = f'"{safe_val}"@{lang}'
        else:
            o = f'"{safe_val}"'
    
    return f"{s} {p} {o} ."


In [20]:

def insert_batch(target_url, triples, named_graph=None):
    if not triples:
        return False

    if named_graph:
        graph_clause = f"GRAPH <{named_graph}>"
    else:
        graph_clause = ''    

    ##  Double Braces in f-string: 
    # Python converts {{ to a single { in the final string.

    # chr(10) represents the Line Feed (LF) character (ASCII value 10)
    # and generates the newline character (\n)

    triples_block = "\n".join(triples)

    query = f"""
    INSERT DATA {{
        {graph_clause} {{
        {triples_block}
        }}
    }}
    """.strip()

#    print(query[:600], "\n\n", query[-600:])




    try:
        r = requests.post(
        TARGET_URL,
        auth=HTTPBasicAuth(TARGET_USER, TARGET_PASS),
        headers={"Content-Type": "application/sparql-update"},
        data=query
        )
        print('Code: ', r.status_code)
        return True
    except Exception as e:
        print(f"✗ Error: {e}")
        return False

In [21]:
print(TARGET_URL)

https://swiss-elites.lod4hss.cloud/wisski/endpoint/default/statements


In [22]:
# Batch Settings
BATCH_SIZE = 30000
SLEEP_SECONDS = 0.5  # Pause between batches to avoid overwhelming the server

In [26]:

offset = 0
total_migrated = 0
batch_count = 0

while True:
    # 1. Fetch Batch
    print(f"Fetching batch starting at offset {offset}...")
#    bindings = fetch_batch(SOURCE_URL, offset, BATCH_SIZE, source_auth)
    bindings = fetch_batch(SOURCE_URL, offset, BATCH_SIZE)
   
    if not bindings:
        print("No more data found. Migration complete.")
        break
    
    # 2. Format Triples
    triple_strings = [format_triple(b) for b in bindings]
    #print("\n".join(triple_strings[:5]))
    
    # 3. Insert Batch
    print(f"Inserting {len(triple_strings)} triples...")
#    success = insert_batch(sparql_insert, triple_strings, target_auth)
    success = insert_batch(sparql_insert, triple_strings, TARGET_GRAPH)

    if success:
        count = len(triple_strings)
        total_migrated += count
        batch_count += 1
        print(f"✓ Batch {batch_count} successful. Total migrated: {total_migrated}")
    else:
        print("✗ Batch failed. Stopping to prevent data inconsistency.")
        break
    
    # Prepare for next iteration
    offset += BATCH_SIZE
    
    # Rate limiting (polite pause)
    time.sleep(SLEEP_SECONDS)

print(f"--- Summary ---")
print(f"Total triples migrated: {total_migrated}")
print(f"Total batches processed: {batch_count}")

Fetching batch starting at offset 0...
Inserting 30000 triples...
INSERT DATA {
        GRAPH <https://swiss-elites.lod4hss.cloud/resource/> {
        <http://elites_suisses/resource/p100511> <http://www.w3.org/2002/07/owl#sameAs> <http://www.wikidata.org/entity/Q15455792> .
<http://elites_suisses/resource/p100941> <http://www.w3.org/2002/07/owl#sameAs> <http://www.wikidata.org/entity/Q15807949> .
<http://elites_suisses/resource/p101194> <http://www.w3.org/2002/07/owl#sameAs> <http://www.wikidata.org/entity/Q78066642> .
<http://elites_suisses/resource/p101316> <http://www.w3.org/2002/07/owl#sameAs> <http://www.wikidata.org/entity/Q1439440> .
<http://elites_s 

 d4hss.cloud/resource/bir_63836> <https://sdhss.org/ontology/shortcuts/P1> "1795"^^<http://www.w3.org/2001/XMLSchema#integer> .
<https://swiss-elites.lod4hss.cloud/resource/bir_63836> <http://www.cidoc-crm.org/cidoc-crm/P96> <https://swiss-elites.lod4hss.cloud/resource/p100821> .
<https://swiss-elites.lod4hss.cloud/resource/bir_6

## Inspect insert

In [1]:
query="""
SELECT ?graph_uri (COUNT(*) AS ?triples_count)
WHERE {
  GRAPH ?graph_uri {
    ?s ?p ?o
  }
}
GROUP BY ?graph_uri
ORDER BY DESC(?triples_count)
"""

In [46]:
query="""
SELECT * 
WHERE { <http://example.org/person/john> ?p ?o }
"""

In [117]:
target_select = SPARQLWrapper(TARGET_URL_SELECT)
target_select.setMethod("GET") # Reset to GET for reading
target_select.setQuery(query)
target_select.setReturnFormat(JSON)
print(target_select.query().convert())

{'head': {'vars': ['graph_uri', 'triples_count']}, 'results': {'bindings': [{'triples_count': {'datatype': 'http://www.w3.org/2001/XMLSchema#integer', 'type': 'literal', 'value': '304450'}, 'graph_uri': {'type': 'uri', 'value': 'http://elites-suisses.org'}}]}}


In [17]:
data = fetch_batch_quads(SOURCE_GRAPH, offset, BATCH_SIZE)

NameError: name 'fetch_batch_quads' is not defined

In [ ]:
print(data[:200])

In [ ]:
v

In [24]:

def insert_batch(sparql_insert, triples, named_graph=None):
    if not triples:
        return False

    if named_graph:
        graph_clause = f"GRAPH <{named_graph}>"
    else:
        graph_clause = ''    

    ##  Double Braces in f-string: 
    # Python converts {{ to a single { in the final string.

    # chr(10) represents the Line Feed (LF) character (ASCII value 10)
    # and generates the newline character (\n)

    triples_block = "\n".join(triples)

    query = f"""
    INSERT DATA {{
        {graph_clause} {{
        {triples_block}
        }}
    }}
    """.strip()

    print(query[:600], "\n\n", query[-600:])

    sparql_insert.setQuery(query)

    try:
        sparql_insert.query()
        return True
    except Exception as e:
        print(f"✗ Error: {e}")
        return False

In [25]:
# Setup the endpoint
sparql_insert = SPARQLWrapper(TARGET_URL)

# not working with RDF4J
# sparql.setHTTPAuth(DIGEST)

sparql_insert.setCredentials(TARGET_USER, TARGET_PASS)

# Configure for UPDATE (INSERT/DELETE)
sparql_insert.setMethod(POST)